# Climate Research Assistant using NLP and Semantic Search

## Project Overview

This project develops a Climate Research Assistant capable of exploring climate science literature using Natural Language Processing (NLP) techniques.

The system processes climate research documents, converts textual information into vector embeddings, stores them in a FAISS vector database, and retrieves the most relevant document sections based on user queries.

The project demonstrates practical applications of:

- Natural Language Processing (NLP)
- Semantic Search
- Text Embeddings
- Vector Databases
- Information Retrieval
- Climate Analytics

---
``

In [2]:
!pip install -q langchain langchain-community langchain-text-splitters pypdf faiss-cpu sentence-transformers

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


def build_vector_db(pdf_path):

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    vectorstore.save_local("faiss_index")

/tmp/ipykernel_1103/2782135802.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Loading Climate Research Documents

Scientific climate reports are typically available in PDF format.

The PDF document is loaded and converted into machine-readable text using the PyPDF document loader.

### Input Document

For this project, climate-related documents such as:

- IPCC Reports
- ERA5 Documentation
- Climate Change Assessments
- Monsoon Research Papers

can be used as knowledge sources.

The extracted text forms the basis for the semantic retrieval pipeline.

In [11]:
build_vector_db('/content/sample_data/IPCC_AR6_WGI_SPM.pdf')

/tmp/ipykernel_1103/2782135802.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Developing the User Interface

A lightweight web interface is built using Streamlit.

The application allows users to enter questions in natural language and retrieve relevant information from climate research documents.

### Example Questions

- What causes monsoon variability?
- How are extreme rainfall events changing?
- What evidence supports climate change?
- What are the impacts of greenhouse gases?

The interface provides an accessible way to explore scientific literature without manually searching through hundreds of pages.

In [12]:
%%writefile app.py
import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

st.title("Climate Research Assistant")

question = st.text_input(
    "Ask a question about climate papers"
)

if question:
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    db = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

    docs = db.similarity_search(question, k=3)

    for doc in docs:
        st.write(doc.page_content)

# --- Code to run the Streamlit app and expose it in Google Colab ---
# Run this in a new cell or uncomment the lines below to start the server:
# !npm install -g localtunnel
# !streamlit run app.py &>/dev/null &
# !urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
# !npx localtunnel --port 8501

Overwriting app.py


## Deploying the Application with Cloudflare Tunnel

Google Colab environments do not provide a publicly accessible URL by default. To enable external access to the Streamlit application, Cloudflare Tunnel is used.

Cloudflare creates a secure tunnel between the local Streamlit server running in Colab and a temporary public URL.

### Why Cloudflare Tunnel?

- No port forwarding required
- Secure public access
- More reliable than LocalTunnel
- Better compatibility with Streamlit applications
- Handles JavaScript assets correctly

### Deployment Workflow

```text
Streamlit App
      │
      ▼
Local Server (Port 8501)
      │
      ▼
Cloudflare Tunnel
      │
      ▼
Public HTTPS URL
      │
      ▼
User Accesses Application
```

### Process

1. Download the Cloudflare Tunnel executable.
2. Grant execution permissions.
3. Launch the Streamlit application locally.
4. Create a secure Cloudflare tunnel.
5. Access the generated public URL from any browser.

### Outcome

The application becomes publicly accessible through a temporary HTTPS URL generated by Cloudflare, allowing users to interact with the Climate Research Assistant without installing any software locally.

In [20]:
!pip install streamlit localtunnel

ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel


In [21]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [22]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [23]:
!./cloudflared tunnel --url http://localhost:8501

2026-10-07T10:35:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-07T10:35:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-07T10:36:01Z INF +--------------------------------------------------------------------------------------------+
2026-10-07T10:36:01Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-07T10:36:01Z INF |  https://effectively-phpbb-charter-sunny.trycloudflare